In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2003_2004"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "HUQ_C.xpt"
csv_file = DATA_DIR / "HUQ_C_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

# Keep untouched source for fidelity comparison
source_df = df.copy()


print("=" * 75)
print("NHANES 2003-2004 HUQ_C XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

print("\nVariables:")
print(df.columns.tolist())


# ============================================================
# 3. EXPECTED RELEASED COLUMNS
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "HUQ010",
    "HUQ020",
    "HUQ030",
    "HUQ040",
    "HUQ050",
    "HUQ060",
    "HUQ071",
    "HUD080",
    "HUQ082",
    "HUQ084",
    "HUQ086",
    "HUQ088A",
    "HUQ088B",
    "HUQ088C",
    "HUQ090"
]


if df.columns.tolist() == EXPECTED_COLUMNS:
    print("\nPASS: Column names/order match HUQ_C.")
else:
    raise ValueError(
        "HUQ_C column names/order do not match expected structure."
    )


# ============================================================
# 4. KNOWN PANDAS/XPT ZERO ARTIFACT
# ============================================================

TINY_VALUE = np.float64(
    5.397605346934028e-79
)

numeric_cols = df.select_dtypes(
    include=[np.number]
).columns


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)

zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


print("\n--- ORIGINAL XPT ZERO CHECK ---")

print(
    "True numeric zeros:",
    f"{int(zero_counts.sum()):,}"
)

print(
    "Tiny-value occurrences:",
    f"{int(tiny_counts.sum()):,}"
)


if tiny_counts.sum() > 0:

    print("\nTiny values by variable:")

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


# ============================================================
# 5. CDC-VALIDATED ZERO VARIABLES
# ============================================================

#
# HUQ050:
# CDC explicitly reports:
#     0 = None
#     count = 1,250
#
# HUQ084:
# CDC permits range:
#     0 - 180
#
# In THIS XPT pandas reads one HUQ084 zero
# as the tiny-value artifact.
#

ZERO_ALLOWED_COLUMNS = [
    "HUQ050",
    "HUQ084"
]


# ============================================================
# 6. SAFETY CHECK
# ============================================================

actual_tiny_columns = set(
    tiny_counts[
        tiny_counts > 0
    ].index
)

approved_zero_columns = set(
    ZERO_ALLOWED_COLUMNS
)


unexpected_tiny = (
    actual_tiny_columns
    -
    approved_zero_columns
)


if unexpected_tiny:

    raise ValueError(
        "STOP: Tiny values found in variables "
        "not approved for zero restoration:\n"
        + str(sorted(unexpected_tiny))
    )


print(
    "\nPASS: Tiny values occur only in "
    "CDC zero-permitted variables."
)


# ============================================================
# 7. RESTORE TRUE ZEROS
# ============================================================

print("\n--- ZERO RESTORATION ---")


for col in ZERO_ALLOWED_COLUMNS:

    count = int(
        df[col]
        .eq(TINY_VALUE)
        .sum()
    )

    print(
        f"{col}: {count:,} tiny values -> 0"
    )

    df.loc[
        df[col].eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 8. VALIDATE ZERO COUNTS
# ============================================================

print("\n--- ZERO COUNTS AFTER RESTORATION ---")


huq050_zeros = int(
    df["HUQ050"]
    .eq(0)
    .sum()
)

huq084_zeros = int(
    df["HUQ084"]
    .eq(0)
    .sum()
)


print(
    "HUQ050 zeros:",
    huq050_zeros
)

print(
    "HUQ084 zeros:",
    huq084_zeros
)


if huq050_zeros != 1250:

    raise ValueError(
        "HUQ050 should contain exactly 1,250 zeros."
    )


if huq084_zeros != 1:

    raise ValueError(
        "HUQ084 expected one restored zero in this XPT."
    )


print(
    "PASS: Zero restoration counts correct."
)


# ============================================================
# 9. CHECK FOR GENUINE DECIMAL VALUES
# ============================================================

#
# IMPORTANT FIDELITY RULE:
#
# Never round a genuine decimal.
#
# First inspect every source value.
# If a genuine fractional value exists, STOP.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = df[col].dropna()

    if values.empty:
        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values)
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"\nGenuine decimal values found in {col}:"
        )

        print(
            values[
                fractional_mask
            ].head(20)
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine decimal values detected. "
        "They will NOT be rounded:\n"
        + str(fractional_columns)
    )


print(
    "PASS: No genuine fractional values exist "
    "in this HUQ_C XPT."
)


# ============================================================
# 10. CLEAN SEQN
# ============================================================

seqn = df["SEQN"].dropna()


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn)
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .round()
    .astype("Int64")
)


print("\n--- SEQN ---")

print(
    "Missing:",
    int(df["SEQN"].isna().sum())
)

print(
    "Duplicates:",
    int(df["SEQN"].duplicated().sum())
)

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)


# ============================================================
# 11. FORMAT WHOLE-NUMBER HUQ VARIABLES
# ============================================================

#
# We already verified there are NO real decimal values.
#
# Therefore:
#
# 1.0   -> 1
# 180.0 -> 180
# 0.0   -> 0
#
# Missing stays missing.
#

HUQ_COLUMNS = [
    col
    for col in df.columns
    if col != "SEQN"
]


for col in HUQ_COLUMNS:

    df[col] = (
        df[col]
        .round()
        .astype("Int64")
    )


print(
    "\nPASS: HUQ codes/counts formatted "
    "without unnecessary .0."
)


# ============================================================
# 12. CDC STRUCTURE CHECK
# ============================================================

print("\n--- CDC STRUCTURE CHECK ---")


if len(df) == 10122:

    print(
        "PASS: Row count = 10,122"
    )

else:

    raise ValueError(
        f"Unexpected row count: {len(df)}"
    )


if len(df.columns) == 16:

    print(
        "PASS: Column count = 16"
    )

else:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


# ============================================================
# 13. CDC EXACT CATEGORICAL FREQUENCIES
# ============================================================

EXPECTED = {

    "HUQ010": {
        "counts": {
            1:3273,
            2:2535,
            3:2788,
            4:1225,
            5:297,
            7:3,
            9:1
        },
        "missing":0
    },

    "HUQ020": {
        "counts": {
            1:1852,
            2:786,
            3:7004,
            7:1,
            9:2
        },
        "missing":477
    },

    "HUQ030": {
        "counts": {
            1:8999,
            2:1069,
            3:52,
            9:2
        },
        "missing":0
    },

    "HUQ040": {
        "counts": {
            1:2286,
            2:6267,
            3:201,
            4:203,
            5:88,
            9:6
        },
        "missing":1071
    },

    "HUQ050": {
        "counts": {
            0:1250,
            1:2082,
            2:2987,
            3:2551,
            4:615,
            5:630,
            99:7
        },
        "missing":0
    },

    "HUQ060": {
        "counts": {
            1:56,
            2:70,
            3:729,
            4:340,
            5:54,
            9:1
        },
        "missing":8872
    },

    "HUQ071": {
        "counts": {
            1:999,
            2:9118,
            9:5
        },
        "missing":0
    },

    "HUQ082": {
        "counts": {
            1:83,
            2:9561,
            9:1
        },
        "missing":477
    },

    "HUQ086": {
        "counts": {
            1:513,
            2:9128,
            9:4
        },
        "missing":477
    },

    "HUQ088A": {
        "counts": {
            1:441,
            9:1
        },
        "missing":9680
    },

    "HUQ088B": {
        "counts": {
            2:29
        },
        "missing":10093
    },

    "HUQ088C": {
        "counts": {
            3:66
        },
        "missing":10056
    },

    "HUQ090": {
        "counts": {
            1:663,
            2:8124,
            9:5
        },
        "missing":1330
    }
}


print("\n--- CDC FREQUENCY CHECK ---")


frequency_failures = []


for col, spec in EXPECTED.items():

    actual_counts = (
        df[col]
        .value_counts(
            dropna=True
        )
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[col]
        .isna()
        .sum()
    )


    if (
        actual_counts == spec["counts"]
        and
        actual_missing == spec["missing"]
    ):

        print(
            f"PASS: {col}"
        )

    else:

        frequency_failures.append(
            col
        )

        print(
            f"\nFAIL: {col}"
        )

        print(
            "Expected counts:",
            spec["counts"]
        )

        print(
            "Actual counts:",
            actual_counts
        )

        print(
            "Expected missing:",
            spec["missing"]
        )

        print(
            "Actual missing:",
            actual_missing
        )


if frequency_failures:

    raise ValueError(
        "CDC frequency mismatch in:\n"
        + str(frequency_failures)
    )


print(
    "\nPASS: All categorical HUQ variables "
    "match CDC."
)


# ============================================================
# 14. HUD080 RANGE/FREQUENCY CHECK
# ============================================================

#
# CDC:
#
# 1-5 = 978
# 6   = 21
# Missing = 9,123
#

s = df["HUD080"]


regular = int(
    s.between(
        1,
        5
    ).sum()
)

six_plus = int(
    s.eq(6).sum()
)

missing = int(
    s.isna().sum()
)


print("\n--- HUD080 ---")

print(
    "1-5:",
    regular
)

print(
    "6 or more:",
    six_plus
)

print(
    "Missing:",
    missing
)


if (
    regular == 978
    and
    six_plus == 21
    and
    missing == 9123
):

    print(
        "PASS: HUD080 matches CDC."
    )

else:

    raise ValueError(
        "HUD080 does not match CDC."
    )


# ============================================================
# 15. HUQ084 RANGE / MISSING CHECK
# ============================================================

#
# CDC:
#
# 0-180 = 82 valid records
# Missing = 10,040
#
# CDC also notes one response of 600 was
# set to missing.
#

s = df["HUQ084"]


print("\n--- HUQ084 ---")

print(
    "Valid:",
    int(s.notna().sum())
)

print(
    "Missing:",
    int(s.isna().sum())
)

print(
    "Minimum:",
    s.min()
)

print(
    "Maximum:",
    s.max()
)

print(
    "Zero count:",
    int(s.eq(0).sum())
)


if (
    int(s.notna().sum()) == 82
    and
    int(s.isna().sum()) == 10040
    and
    s.min() == 0
    and
    s.max() == 180
):

    print(
        "PASS: HUQ084 matches CDC range/count."
    )

else:

    raise ValueError(
        "HUQ084 does not match CDC."
    )


# ============================================================
# 16. BUILD EXPECTED SOURCE
# ============================================================

#
# The only source-value corrections permitted are:
#
# HUQ050 tiny -> 0
# HUQ084 tiny -> 0
#

expected_df = source_df.copy()


for col in ZERO_ALLOWED_COLUMNS:

    expected_df.loc[
        expected_df[col]
        .eq(TINY_VALUE),
        col
    ] = 0.0


# ============================================================
# 17. VERIFY VALUE FIDELITY BEFORE EXPORT
# ============================================================

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    current_values = (
        df[col]
        .astype("Float64")
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} preserved."
        )

    else:

        raise ValueError(
            f"FAIL: {col} changed unexpectedly."
        )


# ============================================================
# 18. FINAL TINY VALUE CHECK
# ============================================================

remaining_tiny = int(
    df.select_dtypes(
        include=[np.number]
    )
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


print("\n--- FINAL ZERO CHECK ---")

print(
    "Remaining tiny values:",
    remaining_tiny
)


if remaining_tiny == 0:

    print(
        "PASS: No tiny-value artifacts remain."
    )

else:

    raise ValueError(
        "Tiny-value artifacts remain."
    )


# ============================================================
# 19. EXPORT CSV
# ============================================================

#
# IMPORTANT:
#
# No float_format.
#
# We already established that this particular HUQ_C
# contains no genuine fractional measurements.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")
print(csv_file)


# ============================================================
# 20. RE-READ CSV
# ============================================================

csv_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


# ============================================================
# 21. EXACT XPT -> CSV FIDELITY
# ============================================================

print("\n--- XPT -> CSV FIDELITY ---")


value_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy()
    )

    exported_values = (
        csv_df[col]
        .to_numpy()
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if same:

        print(
            f"PASS: {col:<10} matches expected XPT."
        )

    else:

        value_failures.append(
            col
        )

        print(
            f"FAIL: {col} differs."
        )


if value_failures:

    raise ValueError(
        "XPT -> CSV differences in:\n"
        + str(value_failures)
    )


print(
    "\nPASS: Every HUQ_C value preserved "
    "after validated zero restoration."
)


# ============================================================
# 22. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing = (
        csv_df[col]
        .isna()
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            csv_missing
        )
    )


print("\n--- MISSING VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches == 0:

    print(
        "PASS: Missing values preserved exactly."
    )

else:

    raise ValueError(
        "Missing-value positions changed."
    )


# ============================================================
# 23. RAW CSV FORMAT CHECK
# ============================================================

print("\n--- RAW CSV FORMAT CHECK ---")


with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


dot_zero_count = (
    csv_text.count(".0")
)


print(
    "Occurrences of '.0':",
    dot_zero_count
)


if dot_zero_count == 0:

    print(
        "PASS: No unnecessary .0 formatting."
    )

else:

    raise ValueError(
        "Unexpected .0 formatting remains."
    )


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count == 0:

    print(
        "PASS: No scientific notation."
    )

else:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print("\n" + "=" * 75)
print("HUQ_C CONVERSION COMPLETE")
print("=" * 75)

NHANES 2003-2004 HUQ_C XPT -> CSV
Rows: 10,122
Columns: 16

Variables:
['SEQN', 'HUQ010', 'HUQ020', 'HUQ030', 'HUQ040', 'HUQ050', 'HUQ060', 'HUQ071', 'HUD080', 'HUQ082', 'HUQ084', 'HUQ086', 'HUQ088A', 'HUQ088B', 'HUQ088C', 'HUQ090']

PASS: Column names/order match HUQ_C.

--- ORIGINAL XPT ZERO CHECK ---
True numeric zeros: 0
Tiny-value occurrences: 1,251

Tiny values by variable:
HUQ050    1250
HUQ084       1

PASS: Tiny values occur only in CDC zero-permitted variables.

--- ZERO RESTORATION ---
HUQ050: 1,250 tiny values -> 0
HUQ084: 1 tiny values -> 0

--- ZERO COUNTS AFTER RESTORATION ---
HUQ050 zeros: 1250
HUQ084 zeros: 1
PASS: Zero restoration counts correct.

--- FRACTIONAL VALUE CHECK ---
PASS: No genuine fractional values exist in this HUQ_C XPT.

--- SEQN ---
Missing: 0
Duplicates: 0
Minimum: 21005
Maximum: 31126

PASS: HUQ codes/counts formatted without unnecessary .0.

--- CDC STRUCTURE CHECK ---
PASS: Row count = 10,122
PASS: Column count = 16

--- CDC FREQUENCY CHECK ---
P